# Documentazione tecnica del progetto RADAR-AS

## Scopo del progetto

RADAR-AS è un progetto Python che espone una piccola applicazione web basata su **Bottle** per lanciare simulazioni **NetLogo** sulla diffusione di fake news in una rete sociale. Il progetto include tre modalità principali:

1. **Test 1**: misura la virality variando la soglia (`threshold`) e la polarizzazione della rete.
2. **Test 2**: misura la virality variando il numero di nodi e la polarizzazione della rete.
3. **Test SA 1**: introduce un **super-agent** controllato da **Deep Q-Learning** per intervenire sulla diffusione.

Il modello di simulazione vero e proprio vive nel file NetLogo `netlogo/FakeNewsSimulation.nlogo`. Il codice Python ha il compito di:

- raccogliere i parametri dal frontend;
- inizializzare NetLogo;
- eseguire batch di simulazioni;
- salvare CSV, log e grafici;
- restituire i risultati al browser.

---

## Punto di ingresso: `run.py`

`run.py` è il file centrale dell’applicazione. Fa tre cose principali:

- crea il server web Bottle;
- espone gli endpoint HTTP per avviare i test;
- orchestra l’esecuzione dei moduli di simulazione.

### Import principali

- `bottle`: framework web minimale.
- `utils`: logging e trasformazione dei risultati per il frontend.
- `test_general_1`, `test_general_2`: moduli dei test “classici”.
- `TestGeneralParametes`: contenitore dei parametri per i test classici.
- `SuperAgentTestGeneralParametes`: contenitore parametri per il test con super-agent.
- `test_general_sa_1`: modulo del test con reinforcement learning.

### Struttura delle route

#### `POST /submit_test_1`
Avvia il **Test 1**.

Flusso:

1. imposta la risposta come stream (`text/event-stream`);
2. crea un oggetto `TestGeneralParametes`;
3. sposta il path di output in `test_general_results/test_general_1/`;
4. intercetta `stdout` con `Tee` per salvare i log sia su file sia in memoria;
5. legge il JSON dal form:
   - `ticks`
   - `iterations`
   - `opinion_polarization`
   - `network_polarization`
   - `thresholds`
6. converte i campi stringa in numeri o liste numeriche;
7. invia log progressivi al frontend;
8. carica NetLogo tramite `test_general_1.load_sim_model()`;
9. esegue il test con `test_general_1.start_test_1(...)`;
10. trasforma il DataFrame in JSON per il grafico con `utils.setup_data_for_chart(...)`;
11. chiude il workspace NetLogo;
12. restituisce al browser:
   - dati del grafico;
   - grafico in base64;
   - log completo.

#### `POST /submit_test_2`
Avvia il **Test 2**.

Differenze principali rispetto al Test 1:

- usa `test_general_results/test_general_2/` come output;
- riceve anche `nodes`;
- usa un singolo `threshold` invece di una lista di soglie;
- chiama `test_general_2.load_sim_model()` e `test_general_2.start_test_2(...)`;
- genera i dati grafico con label `Nodes`.

#### `POST /submit_test_sa_1`
Avvia il test con **super-agent**.

Input aggiuntivi:

- `warning`
- `node_range_static_b`
- `node_range`
- `choose_method`
- `warning_impact`
- `warning_impact_neutral`
- `sa_delay`

Flusso:

1. crea `SuperAgentTestGeneralParametes`;
2. usa `LogManager` al posto di `Tee`;
3. converte i parametri avanzati;
4. carica il modello NetLogo;
5. addestra il modello di reinforcement learning;
6. testa il comportamento del super-agent;
7. salva CSV e grafico;
8. restituisce i risultati al frontend.

#### Route HTML

- `/` e `/index` → homepage
- `/home` → homepage
- `/results` → pagina risultati
- `/test1_page` → pagina form Test 1
- `/test2_page` → pagina form Test 2
- `/static/<filepath:path>` → file statici (CSS, JS, immagini)

### Osservazioni importanti su `run.py`

- è un **entrypoint applicativo**, non un modulo di libreria;
- usa `Bottle` in modalità `debug=True` su `localhost:8081`;
- le risposte dei test sono restituite come **stream testuale**, consumato dal frontend con `fetch(...).body.getReader()`;
- non c’è separazione tra livello web, business logic e orchestrazione simulazione: `run.py` fa molto lavoro e potrebbe essere rifattorizzato;
- il logging è gestito in modo artigianale tramite redirect di `stdout` o file manuali.

---

## Moduli di simulazione

## `test_general_1.py`

Questo modulo implementa il test che studia la virality al variare di:

- `Network Polarization`
- `Thresholds`

### `load_sim_model()`

- crea un `pynetlogo.NetLogoLink(gui=True)`;
- carica `netlogo/FakeNewsSimulation.nlogo`;
- costruisce un wrapper `NetlogoCommands`;
- restituisce `(netlogo, netlogoCommands)`.

### `start_test_1(...)`

Flusso operativo:

1. legge i parametri da `testParameters`;
2. imposta nel modello NetLogo:
   - opinion polarization;
   - valore iniziale `opinion_metric`;
   - echo chamber fraction;
   - numero nodi;
   - numero totale di tick;
3. per ogni valore di `network_polarization`:
   - aggiorna `P_N` in NetLogo;
4. per ogni `threshold`:
   - aggiorna `teta` in NetLogo;
5. per `number_of_iterations` volte:
   - `setup()`;
   - esegue `go()` per `ticks` step;
   - misura `global cascade fraction`;
6. calcola la `Virality` come frazione di simulazioni con cascata > 0.5;
7. salva un CSV `test_general_1.csv`;
8. genera un PDF e un’immagine base64 del grafico.

### Output prodotto

Nella cartella del test:

- `log.txt`
- `test_general_1.csv`
- `test_1_result.pdf`

### Note critiche

- alcune variabili locali (`opinion_metric_steps`, `total_nodes`, `total_ticks`) vengono lette ma non sono davvero usate nel calcolo;
- il codice usa `TestGeneralParametes.echo_chamber_fraction` e `TestGeneralParametes.nb_nodes` come attributi di classe, non sempre quelli dell’istanza;
- il path del modello è coerente qui (`netlogo/FakeNewsSimulation.nlogo`).

---

## `test_general_2.py`

Questo modulo implementa il test che varia:

- `Network Polarization`
- `Nodes`

con `threshold` fissato.

### `load_sim_model()`

- usa `pynetlogo.NetLogoLink(gui=False)`;
- tenta di caricare il modello da `../../netlogo/FakeNewsSimulation.nlogo`.

### Problema importante

Questo path è sospetto/incoerente rispetto alla struttura reale del progetto. Dal root del progetto il file corretto si trova in `netlogo/FakeNewsSimulation.nlogo`, non due directory sopra. Quindi **`test_general_2.py` rischia di rompersi a seconda della current working directory**.

### `start_test_2(...)`

Flusso:

1. legge i parametri;
2. imposta `P_O` e `teta` nel modello;
3. per ogni `network_polarization`:
   - imposta `P_N`;
4. per ogni numero di nodi:
   - imposta `nb-nodes`;
5. ripete la simulazione `number_of_iterations` volte;
6. misura la virality;
7. salva `test_general_2.csv`;
8. genera `test_2.pdf` e una preview base64.

### Output prodotto

In `test_general_results/test_general_2/`:

- `log.txt`
- `test_general_2.csv`
- `test_2.pdf`

### Note critiche

- variabili lette ma poco o mai usate: `opinion_metric_steps`, `opinion_metric_value`, `global_cascades_means`;
- la funzione `set_treshold` riceve qui un singolo threshold dal form, mentre altrove il nome `thresholds` suggerisce una lista;
- il nome `modelfile` globale in testa al modulo è ridondante rispetto a quello ridefinito nella funzione.

---

## `test_general_sa_1/test_general_sa_1.py`

È il modulo più complesso. Implementa il test con **super-agent** e **Deep Q-Learning**.

### Ruolo del modulo

- inizializza NetLogo;
- costruisce un ambiente tipo Gymnasium;
- configura i parametri del super-agent;
- addestra una rete neurale;
- usa il modello addestrato per decidere le azioni durante la simulazione.

### `load_sim_model()`

- crea `pynetlogo.NetLogoLink(gui=True)`;
- carica `netlogo/FakeNewsSimulation.nlogo`;
- restituisce il wrapper `NetlogoCommands` specializzato per il super-agent.

### `start_test_sa_1(...)`

Flusso sintetico:

1. legge i parametri standard e quelli del super-agent;
2. crea `FakeNewsSimulation(netlogoCommands)`;
3. reimposta numerosi parametri nel modello NetLogo;
4. imposta i criteri di scelta dei nodi influenti (`choose_method`);
5. istanzia `DeepQLearning()`;
6. per ogni `network_polarization` e `threshold`:
   - imposta i valori in NetLogo;
   - addestra il modello con `run_model_training(...)`;
   - esegue la fase di test per `number_of_iterations` simulazioni;
   - durante i tick chiama:
     - `env.step(0)` all’inizio e nei tick normali;
     - `predict_sa_action(...)` ogni `sa_delay` tick;
7. misura la virality;
8. salva `test_general_sa_1.csv`;
9. genera il grafico PDF/base64.

### Azioni del super-agent

Le azioni disponibili sono codificate come:

- `0` → `go`
- `1` → `warning`
- `2` → `reiterate`
- `3` → `static_b`

### Note critiche

- viene usato `sys.path.insert(1, './test_general_sa_1')`, segnale che la struttura dei package non è stata pulita;
- c’è una riassegnazione di `netlogoCommands = NetlogoCommands(netlogo, modelfile)` dopo la creazione dell’ambiente: è poco chiara e può rendere fragile il flusso;
- il `modelfile` dichiarato all’inizio (`test_super_agent/netlogo/...`) non corrisponde poi al path davvero usato in `load_sim_model()`;
- il training RL viene ripetuto dentro i doppi loop di polarizzazione e soglia, quindi il costo computazionale è alto;
- il codice miscela italiano e inglese nei log e nei nomi.

---

## File parametri e wrapper NetLogo

## `test_general_parameters.py`

Contiene:

1. la classe `TestGeneralParametes`;
2. la classe `NetlogoCommands` per i test classici;
3. la funzione `calculate_fraction(values)`.

### `TestGeneralParametes`

Funziona come contenitore mutabile dei parametri, con default iniziali:

- `network_polarization = np.linspace(0, 1, num=13)`
- `opinion_polarization = 0`
- `thresholds = [...]`
- `path = "test_general_results/"`
- `number_of_iterations = 100`
- `echo_chamber_fraction = 0.20`
- `opinion_metric_value = 0.5`
- `opinion_metric_steps = [...]`
- `nb_nodes = 100`
- `total_ticks = 100`
- parametri emotivi: `arousal_threshold`, `arousal`, `valence`, `h_ec`, `h_network`

### `NetlogoCommands`

Espone metodi di lettura e scrittura verso NetLogo:

- letture: agenti attivi A/B, neutrali, totale agenti, tick totali;
- scritture: `P_O`, `P_N`, `teta`, `nb-nodes`, `initial-opinion-metric-value`, `opinion-metric-step`, `echo-chamber-fraction`, `total-ticks`;
- controllo simulazione: `setup()`, `go()`.

### `calculate_fraction(values)`

Conta quanti valori sono `> 0.5` e restituisce la frazione sul totale. È la definizione pratica usata per la `Virality`.

### Note critiche

- il nome della classe contiene un refuso: `Parametes` invece di `Parameters`;
- molti attributi sono definiti a livello di classe e poi trattati quasi come attributi di istanza;
- ci sono setter per parametri emotivi che nei test principali non sembrano essere sfruttati.

---

## `test_general_sa_1/test_general_parameters_sa.py`

Analogo al precedente, ma per il test con super-agent.

### `SuperAgentTestGeneralParametes`

Default principali:

- `network_polarization = np.linspace(0, 1, num=13)`
- `thresholds = [0.270, 0.342, 0.414]`
- `path = "test_general_sa_1/test_general_sa_1_result/"`
- `number_of_iterations = 100`
- `opinion_metric_step = 0.10`
- `warning = True`
- `node_range_static_b = 0.05`
- `node_range = 0.10`
- `choose_method = "degree"`
- `warning_impact = 0.10`
- `warning_impact_neutral = 0.30`
- `sa_delay = 5`

### `NetlogoCommands` del super-agent

Oltre ai metodi base, aggiunge:

- `get_most_influent_a_nodes(...)`
- `get_global_opinion_metric_mean()`
- `set_warning(...)`
- `set_node_range(...)`
- `set_node_range_static_b(...)`
- `set_warning_impact(...)`
- `set_warning_impact_neutral(...)`
- `activate_warning()`
- `activate_reiterate()`
- `activate_static_b_node()`
- `choose_action(choice)`

### Criteria supportati

- `betweenness`
- `page-rank`
- `degree`
- `degree-in-cluster`
- `random`

### Note critiche

- anche qui c’è il refuso `Parametes`;
- il setter `set_network_polarization(self, network_olarization)` contiene un typo nel nome argomento;
- `set_opinion_metric_steps` salva `self.opinion_metric_steps`, ma il default nel file è `opinion_metric_step` singolare: c’è incoerenza concettuale.

---

## Ambiente RL

## `environment/fake_news_diffusion_env.py`

Definisce l’ambiente Gymnasium che collega l’agente RL a NetLogo.

### Funzione del file

- definisce lo spazio delle osservazioni;
- definisce lo spazio delle azioni;
- implementa `reset()` e `step()`;
- calcola reward e transizioni;
- gestisce lo stato del super-agent.

In pratica è l’adattatore tra il simulatore NetLogo e il training Deep Q-Learning.

## `environment/environment_utils.py`

Contiene la logica del reward.

Responsabilità principali:

- memorizzare i valori di cascata globale nel tempo;
- calcolare ricompense diverse in base a:
  - azione eseguita;
  - tick corrente;
  - andamento della cascata;
  - peso associato a warning/reiterate/static/go.

Il codice è piuttosto “manuale”: molte regole sono hard-coded con `match/case` e rami annidati.

### `netlogo/simulation_parameters.py`

Contiene costanti globali di configurazione per il reward:

- `NumberOfTicks = 100`
- `WarningWeight = 10`
- `ReiterateWeight = 3`
- `StaticWeight = 5`
- `GoWeight = 2`

### `netlogo/simulation_controls.py`

Definisce un wrapper alternativo per i comandi NetLogo usati dall’ambiente RL.

Questo file è simile ai wrapper presenti nei file parametri, ma non è il wrapper principale usato da `run.py` per i test classici.

---

## Reinforcement learning

## `test_general_sa_1/deepq_simulation.py`

Implementa un semplice Deep Q-Learning con TensorFlow/Keras.

### Componenti principali

#### `agent(state_shape, action_shape)`
Costruisce una rete neurale fully-connected:

- Dense(24, relu)
- Dense(12, relu)
- Dense(output lineare)

Loss: `Huber()`

Optimizer: `Adam(learning_rate=0.001)`

#### `train(...)`

- usa replay memory (`deque`);
- batch size 128;
- aggiorna i Q-values con formula classica Q-learning;
- addestra il modello principale;
- usa un target network.

#### `run_model_training(...)`

- inizializza `model` e `target_model`;
- avvia training epsilon-greedy;
- aggiorna periodicamente il target model;
- misura il tempo di training.

#### `predict_sa_action(...)`

- usa `target_model.predict(...)`;
- sceglie `argmax` come azione;
- esegue uno step nell’ambiente.

### Note critiche

- l’API Gymnasium è usata in modo non perfettamente uniforme;
- c’è codice non utilizzato o poco utile (`X`, `y`, `total_test_episodes`, ecc.);
- i parametri di training sono hard-coded;
- manca persistenza del modello addestrato su disco;
- il training ricomincia da zero ogni volta.

---

## Utility

## `utils.py`

Contiene due classi di logging e una funzione per i grafici frontend.

### `Tee`

Serve a duplicare `stdout` su:

- terminale;
- file di log;
- buffer in memoria.

Usato in `submit_test_1` e `submit_test_2`.

### `LogManager`

Gestisce un log di testo “manuale” su file con metodi:

- `insert_line(...)`
- `get_contents()`
- `clear_log()`

Usato in `submit_test_sa_1`.

### `setup_data_for_chart(dataframe, dataLabel)`

Trasforma un DataFrame pandas in una struttura JSON per il frontend:

- `thresholds`: valori unici dell’asse/serie
- `datasets`: virality per ciascuna serie
- `network_polarization`: valori asse X

### Nota critica

In fondo al file c’è un `if __name__ == "__main__":` con un path assoluto locale (`/Users/andrea/...`) che è chiaramente residuo di sviluppo e non portabile.

---

## Frontend

## `app/templates/`

Contiene le pagine HTML Bottle:

- `homepage.html`
- `result-page.html`
- `test-page-1.html`
- `test-page-2.html`

### Ruolo

- homepage: navigazione ai test;
- test-page-1: form per Test 1 e super-agent;
- test-page-2: form per Test 2;
- result-page: visualizzazione log e grafico finale.

## `app/static/script/`

### `form_script.js`

Gestisce:

- validazione client-side dei campi;
- toggle degli help/info box;
- apertura/chiusura dei parametri avanzati super-agent.

Regex usate:

- interi positivi
- decimali in [0,1]
- liste di decimali separate da virgola
- liste di interi separate da virgola

### `main_script.js`

È il vero orchestratore lato browser.

Responsabilità:

- leggere i campi del form;
- costruire il JSON di input;
- chiamare gli endpoint `/submit_test_1`, `/submit_test_2`, `/submit_test_sa_1`;
- leggere lo stream di risposta;
- aggiornare in diretta il box dei log;
- salvare in `sessionStorage` i risultati;
- reindirizzare a `/results`.

### `result-page-script.js`

Gestisce la pagina risultati:

- legge i dati da `sessionStorage`;
- mostra il log testuale;
- mostra l’immagine base64 del grafico;
- prepara il download di log e grafico.

### Nota importante sul grafico

Esiste codice per costruire una configurazione chart-like (`prepare_chart_data`), ma la pagina finale usa anche l’immagine statica base64. Questo suggerisce che il rendering grafico JS non sia stato completamente rifinito o che siano rimasti due approcci diversi.

## `app/static/styles/`

Foglio di stile per homepage, form, loading e pagina risultati.

## `app/static/assets/`

Immagini decorative usate dall’interfaccia.

---

## Dipendenze del progetto

Dal `Pipfile` e dagli import reali emergono soprattutto queste dipendenze:

### Backend / server

- `bottle`

### Simulazione e analisi

- `pynetlogo`
- `pandas`
- `numpy`
- `matplotlib`

### Reinforcement learning

- `tensorflow`
- `gymnasium`

### Standard library

- `os`, `sys`, `time`, `json`, `io`, `base64`, `random`, `collections.deque`, `pathlib`

### Dipendenze implicite di sistema

Non stanno nel codice Python, ma per far funzionare davvero il progetto servono anche:

- **Java** (richiesto da NetLogo / pyNetLogo)
- **NetLogo installato** o comunque accessibile a pyNetLogo

### Dipendenze lato frontend

- jQuery via CDN
- probabilmente Bootstrap o CSS custom (dipende dai template)

---

## Struttura delle directory

## Directory sicuramente usate nel flusso principale

### `app/`
Usata dal web server per template, CSS, JS e asset.

### `netlogo/`
Usata per il file `FakeNewsSimulation.nlogo` e per i moduli di supporto del layer RL.

### `environment/`
Usata dal test con super-agent per definire l’ambiente RL.

### `test_general_sa_1/`
Usata direttamente da `run.py`:

- `test_general_sa_1.py`
- `test_general_parameters_sa.py`
- `deepq_simulation.py`
- cartella output `test_general_sa_1_result/`

### file root usati

- `run.py`
- `test_general_1.py`
- `test_general_2.py`
- `test_general_parameters.py`
- `utils.py`
- `Pipfile`
- `Pipfile.lock`
- `README.md`

---

## Directory di output / dati generati

### `test_general_results/`
È una cartella **usata** dal progetto come output dei test classici.

Sottocartelle osservate:

- `test_general_results/test_general_1/`
- `test_general_results/test_general_2/`

Contiene file generati:

- log
- CSV
- PDF

### `test_general_sa_1/test_general_sa_1_result/`
È una cartella **usata** come output del test con super-agent.

---

## Directory probabilmente inutili o non necessarie in produzione

### `backup/`
Questa è la directory più chiaramente “non di produzione”.

Perché sembra inutile nel flusso attuale:

- `run.py` non la importa mai;
- non ci sono riferimenti a `backup` nel codice principale;
- contiene codice vecchio, test storici, file sperimentali, PDF, varianti multiple del progetto;
- occupa molto spazio ma non è collegata all’esecuzione corrente.

Conclusione: **può essere trattata come archivio storico**, non come parte del runtime.

### `__MACOSX/`
È spazzatura tipica degli zip creati su macOS. Non serve al progetto.

### `.idea/`
Configurazione IDE JetBrains/PyCharm. Non serve al runtime.

### `__pycache__/`
Bytecode Python generato automaticamente. Non va versionato in un progetto pulito.

### `.git/`
Storico Git. Non è codice applicativo.

### `.DS_Store`
File macOS inutile per il progetto.

### `test_general_sa_1/netlogo/` e `test_general_sa_1/environment/`
Queste sottocartelle sembrano **duplicati** dei moduli root `netlogo/` e `environment/` oppure materiale di sviluppo locale.

Nel flusso effettivo, `test_general_sa_1/test_general_sa_1.py` importa `from environment...` e `from deepq_simulation...` dopo avere modificato `sys.path`. Questa struttura è fragile, ma la cartella root `environment/` è quella che appare nel flusso principale.

Le copie dentro `test_general_sa_1/` sembrano ridondanti o, quantomeno, fonte di confusione.

### `test_general_sa_1/graphic_builder.ipynb`
Notebook di analisi / plotting manuale. Non fa parte del runtime.

### sottocartelle come `degree_ec/`, `random/`, `environment/` interne a `test_general_sa_1`
Sembrano contenere risultati o materiale di esperimenti. Non risultano richiamate da `run.py`.

---

## Problemi strutturali rilevati

### 1. Path incoerenti

Il caso più evidente è `test_general_2.py`, che usa `../../netlogo/FakeNewsSimulation.nlogo`. Questo può fallire facilmente.

### 2. Nomi incoerenti / refusi

- `Parametes` invece di `Parameters`
- `treshold` invece di `threshold`
- mix di `threshold` e `thresholds`
- variabile `network_olarization` con typo

### 3. Uso improprio di attributi di classe

Le classi parametri sembrano pensate come istanze configurabili, ma molti valori restano definiti a livello di classe e vengono usati in quel modo.

### 4. Package structure fragile

L’uso di `sys.path.insert(...)` è un campanello d’allarme. Significa che la struttura dei moduli non è ben organizzata come package Python.

### 5. Duplicazione di wrapper NetLogo

Esistono più `NetlogoCommands` in posti diversi:

- `test_general_parameters.py`
- `test_general_sa_1/test_general_parameters_sa.py`
- `netlogo/simulation_controls.py`

Questo crea ridondanza e rischio di divergenze.

### 6. Logging non uniforme

- Test 1 e 2 usano `Tee`
- Test SA usa `LogManager`

Sarebbe meglio un’unica strategia.

### 7. Mescolanza di italiano e inglese

Il progetto è leggibile, ma meno coerente da mantenere.

### 8. Output e codice mescolati

Le cartelle di output (`test_general_results`, `test_general_sa_1_result`) stanno dentro il repository assieme al codice sorgente.

---

## Flusso completo del progetto

1. L’utente apre `/`.
2. Sceglie uno dei test dal frontend.
3. Il browser invia un POST a `run.py`.
4. `run.py` valida/converte i parametri.
5. Viene avviato NetLogo via `pynetlogo`.
6. Il modulo Python dedicato esegue le iterazioni di simulazione.
7. I risultati vengono salvati in CSV/PDF/log.
8. Il backend streamma gli aggiornamenti al frontend.
9. Il frontend salva tutto in `sessionStorage`.
10. La pagina `/results` mostra grafico e log.

---

## Cosa tenere a mente prima di modificarlo

### Parti sensibili

- path verso `FakeNewsSimulation.nlogo`
- interfaccia Bottle ↔ frontend streaming
- wrappers `NetlogoCommands`
- ambiente RL e reward shaping
- cartelle output create dinamicamente

### Parti che puoi considerare secondarie o rimovibili dal runtime

- `backup/`
- `__MACOSX/`
- `.idea/`
- `__pycache__/`
- `.DS_Store`
- notebook `.ipynb`
- eventuali copie duplicate sotto `test_general_sa_1/`

---

## Valutazione finale

Il progetto è funzionale ma ha una struttura da **prototipo di ricerca**, non da prodotto rifinito.

I punti forti sono:

- idea architetturale chiara;
- integrazione tra web app, NetLogo e RL;
- salvataggio risultati già presente;
- frontend sufficiente per provare i test.

I limiti principali sono:

- path e import fragili;
- molte duplicazioni;
- naming incoerente;
- presenza di directory storiche o inutili nel repository;
- accoppiamento forte tra UI, logica e simulazione.

Se il tuo obiettivo è modificarlo, il primo passo consigliato sarebbe:

1. separare il codice attivo da backup/esperimenti;
2. sistemare i path del modello NetLogo;
3. unificare i wrapper NetLogo;
4. isolare i parametri in dataclass vere;
5. consolidare l’ambiente RL in una sola posizione.

